# z015 - Canaritos (punto 1.6 del profesor)

**Idea:** agregar 50 variables **totalmente al azar** ("canaritos"). Una variable real que el modelo usa
**menos que el mejor canarito** no aporta más que el ruido → se elimina.

1. Entrenar con variables + 50 canaritos (10 semillas) y promediar la importancia (gain).
2. Eliminar las variables con importancia menor que la del canarito más importante.
3. Comparar "Actual" vs "Sin las que pierden con canaritos" (10 semillas).

Ojo: la selección y la evaluación usan los mismos meses → el resultado puede ser algo optimista.
Trial 34, 538 árboles, train 202104, valid 202106.

In [1]:
# --- 1. Librerías y parámetros comunes ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MES_VALID = 202106
SEMILLAS  = [501419, 623353, 786349, 834569, 946331,   # las tuyas
             111119, 222227, 333331, 444443, 555557]  # 5 más (las mismas del ensamble)
N_PARTICIONES = 50        # Shuffle Split del test: particiones al azar del 50% de 202106

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2. Cargar datos y lags (igual que z003) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria", "clase_binaria2"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
valid = df[df["foto_mes"] == MES_VALID]
# La ganancia SIEMPRE se calcula con BAJA+2 (aunque se entrene con otro target)
ganancia_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
print("Variables:", len(variables))

Variables: 761


In [3]:
# --- 3. Funciones de evaluación ---
def metricas(prob, mascara=None):
    # Ganancia acumulada ordenando de mayor a menor probabilidad (opcional: solo una parte de los clientes)
    g = ganancia_cliente if mascara is None else ganancia_cliente[mascara]
    p = prob if mascara is None else prob[mascara]
    acum = np.cumsum(g[np.argsort(p)[::-1]])
    n = int(np.argmax(acum)) + 1
    return acum[n - 1] / 1e6, n, acum[min(11_999, len(acum) - 1)] / 1e6

def resumen(probs_por_version):
    """probs_por_version = {version: [prob_semilla1, ..., prob_semilla5]}"""
    # a) Por semilla: ganancia máxima, mejor N y top 12.000
    filas = []
    for v, lista in probs_por_version.items():
        for s, pr in zip(SEMILLAS, lista):
            g, n, g12 = metricas(pr)
            filas.append({"version": v, "semilla": s, "ganancia_max_M": round(g, 1), "mejor_n": n, "top12000_M": round(g12, 1)})
    por_semilla = pd.DataFrame(filas)
    tabla = por_semilla.groupby("version", sort=False)[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1)

    # b) Shuffle Split del test: ensamble de las 5 semillas evaluado en 50 particiones del 50% de 202106
    rng = np.random.default_rng(SEMILLAS[0])
    ens = {v: np.mean(lista, axis=0) for v, lista in probs_por_version.items()}
    part = []
    for i in range(N_PARTICIONES):
        mascara = rng.random(len(ganancia_cliente)) < 0.5
        part.append({v: metricas(pr, mascara)[0] for v, pr in ens.items()})
    part = pd.DataFrame(part)
    tabla[("shuffle_split", "mean")] = part.mean().round(1)
    tabla[("shuffle_split", "std")] = part.std().round(1)
    tabla[("shuffle_split", "veces_mejor")] = part.idxmax(axis=1).value_counts().reindex(tabla.index).fillna(0).astype(int)
    tabla[("ensamble_5", "ganancia_max_M")] = [round(metricas(ens[v])[0], 1) for v in tabla.index]
    return por_semilla, tabla

def anotar(tabla, prefijo, meses_train, extra=""):
    ruta_res = "resultados_experimentos.csv"
    res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
    res = res[~res["descripcion"].str.startswith(prefijo)]   # sin duplicar si se vuelve a correr
    for v, r in tabla.iterrows():
        nueva = {
            "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
            "descripcion": (f"{prefijo} {v}: prom. {len(SEMILLAS)} semillas; shuffle split {r[('shuffle_split', 'mean')]} "
                            f"(mejor en {r[('shuffle_split', 'veces_mejor')]}/{N_PARTICIONES}){extra}"),
            "meses_train": str(meses_train), "mes_valid": MES_VALID, "semilla": str(SEMILLAS),
            "params": str(params), "mejor_n": int(r[("mejor_n", "mean")]),
            "ganancia_max_M": r[("ganancia_max_M", "mean")], "ganancia_top12000_M": r[("top12000_M", "mean")],
        }
        res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)
    res["n_enviados"] = res["n_enviados"].astype("Int64")
    res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
    print("Anotado en", ruta_res)

In [4]:
# --- 4. Entrenar con canaritos y medir importancias ---
rng = np.random.default_rng(SEMILLAS[0])
N_CANARITOS = 50
train = df[df["foto_mes"] == 202104].copy()
canaritos = [f"canarito_{i}" for i in range(N_CANARITOS)]
for c in canaritos:
    train[c] = rng.random(len(train)).astype("float32")

datos = lgb.Dataset(train[variables + canaritos], label=train["clase_binaria"])
imp = np.mean([lgb.train({**params, "seed": s}, datos, num_boost_round=ARBOLES).feature_importance("gain")
               for s in SEMILLAS], axis=0)
imp = pd.Series(imp, index=variables + canaritos).sort_values(ascending=False)
umbral = imp[canaritos].max()
eliminadas = [v for v in variables if imp[v] < umbral]
quedan = [v for v in variables if imp[v] >= umbral]
print("Puesto del mejor canarito:", list(imp.index).index(imp[canaritos].idxmax()) + 1, "de", len(imp))
print("Variables que quedan:", len(quedan), "| eliminadas:", len(eliminadas))
pd.Series(eliminadas).to_csv("z015_eliminadas.csv", index=False, header=False)

Puesto del mejor canarito: 39 de 811
Variables que quedan: 38 | eliminadas: 723


In [5]:
# --- 4b. Comparar Actual vs sin las eliminadas ---
train = df[df["foto_mes"] == 202104]
probs = {}
for version, vars_v in [("Actual", variables), ("Sin las que pierden con canaritos", quedan)]:
    datos = lgb.Dataset(train[vars_v], label=train["clase_binaria"])
    probs[version] = [lgb.train({**params, "seed": s}, datos, num_boost_round=ARBOLES).predict(valid[vars_v])
                      for s in SEMILLAS]
    print(version, len(vars_v), "variables, listo")

Actual 761 variables, listo


Sin las que pierden con canaritos 38 variables, listo


In [6]:
# --- 5. Comparación ---
por_semilla, tabla = resumen(probs)
print(por_semilla.pivot(index="semilla", columns="version", values="ganancia_max_M"))
tabla

version  Actual  Sin las que pierden con canaritos
semilla                                           
111119    396.8                              371.5
222227    396.8                              378.0
333331    403.6                              372.7
444443    396.9                              372.1
501419    400.5                              381.1
555557    401.0                              372.8
623353    397.2                              376.7
786349    401.4                              375.7
834569    397.2                              376.4
946331    401.8                              373.3


ganancia_max_M      top12000_M       \
                                            mean  std       mean  std   
version                                                                 
Actual                                     399.3  2.6      391.7  3.7   
Sin las que pierden con canaritos          375.0  3.1      368.0  5.5   

                                   mejor_n         shuffle_split        \
                                      mean     std          mean   std   
version                                                                  
Actual                             13230.9  1224.2         198.9  11.8   
Sin las que pierden con canaritos  12846.9   958.9         186.4  12.1   

                                                  ensamble_5  
                                  veces_mejor ganancia_max_M  
version                                                       
Actual                                     50          400.5  
Sin las que pierden con canaritos           0          374.8

In [7]:
# --- 6. Anotar ---
anotar(tabla, "z015 canaritos", [202104])

Anotado en resultados_experimentos.csv


## Conclusiones (08/10)
- El mejor canarito quedó en el puesto 39 de 811 → con el criterio "menos que el mejor canarito" **se eliminaron 723 variables** (quedaron 38).
- Resultado: **375,0 M** vs 399,3 → **−24 M**. El criterio fue demasiado agresivo (las variables que suman poco, sumadas, aportan).
- Si se quisiera reintentar: umbral más suave (por ejemplo, la mediana de los canaritos).